# Zenith CCTV action recognition

Train a three-class prototype that flags **suspected physical fights**, **falls**, and **normal movement** in CCTV clips. YOLO11 Pose extracts body keypoints, ByteTrack follows people, and a small Transformer learns movement over time.

1. Open this notebook in [Google Colab](https://colab.research.google.com/).
2. Go to **Runtime → Change runtime type → T4 GPU**.
3. Choose **Runtime → Run all**.

The notebook downloads the public CCTV Action Recognition dataset from Kaggle (about 542 MB), prepares the clips, trains, evaluates, and runs a demo on a held-out clip. The default Kaggle download uses a public endpoint and needs no API key. Pose extraction is the longest step and is cached for reruns in the same runtime. Colab's runtime storage is temporary, so download the trained model after the run.

Dataset source is the [CCTV Action Recognition Dataset](https://www.kaggle.com/datasets/jonathannield/cctv-action-recognition-dataset). The model predicts one action at a time. It is a prototype for human-reviewed alerts.

## 1. Settings and setup

In [ ]:
# @title Training settings
EPOCHS = 20 # @param {type:"integer"}
DOWNLOAD_CHECKPOINT = True # @param {type:"boolean"}
DEMO_VIDEO = "" # @param {type:"string"}
# Leave DEMO_VIDEO blank to use an automatic held-out example.
# To test your own clip after training, upload it in the Colab Files panel,
# set DEMO_VIDEO to its path (for example /content/my_clip.mp4),
# rerun this settings cell, and then rerun only the demo cell.

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime → Change runtime type → T4 GPU, then run this notebook again.")
if EPOCHS < 1:
    raise ValueError("EPOCHS must be at least 1.")
print(torch.cuda.get_device_name(0))

In [ ]:
%pip -q install "ultralytics>=8.3,<9" "lap>=0.5.12" scikit-learn tqdm

## 2. Pipeline code

These cells create the training and inference modules in the Colab runtime.

In [ ]:
%%writefile common.py
"""Shared constants and helpers for the CCTV action pipeline."""
import numpy as np

CLASSES = ["normal", "violence", "fall"]
T, P, K = 32, 4, 17  # window length (frames), max people per window, keypoints
# COCO left/right swap order for horizontal-flip augmentation
FLIP = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]


def frame_tracks(r):
    """Ultralytics pose-track result -> {track_id: (17,3) [x,y,conf] normalized}."""
    if r.boxes is None or r.boxes.id is None or r.keypoints is None:
        return {}
    ids = r.boxes.id.int().cpu().tolist()
    xy = r.keypoints.xyn.cpu().numpy()
    cf = r.keypoints.conf
    cf = cf.cpu().numpy() if cf is not None else np.ones(xy.shape[:2], np.float32)
    kp = np.concatenate([xy, cf[..., None]], -1).astype(np.float32)
    return {i: kp[n] for n, i in enumerate(ids)}


def build_window(frames):
    """list of {id: (17,3)} (<= T long) -> array (P,T,17,3), or None if nobody visible."""
    frames = (list(frames) + [{}] * T)[:T]
    counts = {}
    for f in frames:
        for i in f:
            counts[i] = counts.get(i, 0) + 1
    if not counts:
        return None
    top = sorted(counts, key=counts.get, reverse=True)[:P]
    x = np.zeros((P, T, K, 3), np.float32)
    for p, i in enumerate(top):
        for t, f in enumerate(frames):
            if i in f:
                x[p, t] = f[i]
    return x

TARGET_FPS = 10
POSE_WEIGHTS = "yolo11n-pose.pt"


def track_video(model, path, capture_info):
    """Yield tracked frames at roughly TARGET_FPS for a local video file."""
    import cv2

    capture = cv2.VideoCapture(str(path))
    try:
        fps = capture.get(cv2.CAP_PROP_FPS)
    finally:
        capture.release()
    if not np.isfinite(fps) or fps <= 0:
        return
    stride = max(1, round(fps / TARGET_FPS))
    capture_info["sample_fps"] = fps / stride
    yield from model.track(
        source=str(path), stream=True, persist=False, vid_stride=stride,
        tracker="bytetrack.yaml", device=0, verbose=False,
    )


def action_window(frames):
    """Build a window with enough observations of at least one track."""
    window = build_window(frames)
    if window is None:
        return None
    visible_frames = (window[..., 2] > 0).any(axis=-1).sum(axis=-1)
    return window if visible_frames.max() >= 4 else None

In [ ]:
%%writefile model.py
"""Skeleton-sequence transformer: per-person temporal encoder + max-pool over people."""
import torch
import torch.nn as nn
from common import T


def feats(x):
    """x: (B,P,T,17,3) -> (B,P,T,89) motion-aware features."""
    xy, c = x[..., :2], x[..., 2]
    present = (c.sum(-1) > 0).float()                       # B,P,T
    hip = (xy[..., 11, :] + xy[..., 12, :]) / 2             # B,P,T,2
    rel = (xy - hip.unsqueeze(-2)) * present[..., None, None]
    pair = present[:, :, 1:] * present[:, :, :-1]
    vel = torch.zeros_like(xy)
    vel[:, :, 1:] = (xy[:, :, 1:] - xy[:, :, :-1]) * pair[..., None, None]
    hv = torch.zeros_like(hip)
    hv[:, :, 1:] = (hip[:, :, 1:] - hip[:, :, :-1]) * pair[..., None]
    return torch.cat([rel.flatten(-2), vel.flatten(-2), hip, hv, c], -1)


class Net(nn.Module):
    def __init__(self, n_classes, f_dim=89, d=128):
        super().__init__()
        self.inp = nn.Linear(f_dim, d)
        self.pos = nn.Parameter(torch.zeros(1, T, d))
        layer = nn.TransformerEncoderLayer(d, 4, 256, 0.1, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, 2)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_classes))

    def forward(self, x):
        B, P = x.shape[:2]
        pres = (x[..., 2].sum(-1) > 0).float()              # B,P,T
        h = self.enc(self.inp(feats(x).flatten(0, 1)) + self.pos)
        m = pres.flatten(0, 1).unsqueeze(-1)
        h = (h * m).sum(1) / m.sum(1).clamp(min=1)          # mean over visible frames
        h = h.view(B, P, -1)
        person = (pres.sum(-1) > 0).unsqueeze(-1)
        h = h.masked_fill(~person, -1e4).max(1).values      # max over people
        return self.head(h)

In [ ]:
%%writefile train.py
"""Train the skeleton action classifier.  python train.py --data data --epochs 40"""
import argparse
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from common import CLASSES, FLIP
from model import Net


class DS(torch.utils.data.Dataset):
    def __init__(self, root, split, aug):
        self.files = sorted(Path(root, split).glob("*.npz")); self.aug = aug

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        with np.load(self.files[i], allow_pickle=False) as d:
            x, y = d["x"], int(d["y"])
        if self.aug:
            if np.random.rand() < 0.5:  # horizontal flip with L/R joint swap
                x = x[:, :, FLIP]
                x[..., 0] = np.where(x[..., 2] > 0, 1 - x[..., 0], 0)
            x[..., :2] += np.random.randn(*x[..., :2].shape).astype(np.float32) * 0.005 * (x[..., 2:3] > 0)
        return torch.from_numpy(x), y


def macro_f1(y, p, n):
    f = []
    for c in range(n):
        tp = ((y == c) & (p == c)).sum(); fp = ((y != c) & (p == c)).sum(); fn = ((y == c) & (p != c)).sum()
        f.append(2 * tp / max(2 * tp + fp + fn, 1))
    return float(np.mean(f))


def evaluate(net, dl, dev):
    net.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in dl:
            ps.append(net(x.to(dev)).argmax(-1).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data"); ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--bs", type=int, default=64); ap.add_argument("--out", default="action.pt")
    a = ap.parse_args()
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tr, va = DS(a.data, "train", True), DS(a.data, "val", False)
    tl = torch.utils.data.DataLoader(tr, a.bs, shuffle=True, num_workers=2)
    vl = torch.utils.data.DataLoader(va, a.bs, num_workers=2)

    labels = []
    for path in tr.files:
        with np.load(path, allow_pickle=False) as example:
            labels.append(int(example["y"]))
    counts = np.bincount(labels, minlength=len(CLASSES))
    print("Train clips per class:", dict(zip(CLASSES, counts.tolist())), flush=True)
    w = torch.tensor(counts.sum() / np.maximum(counts, 1) / len(CLASSES), dtype=torch.float32).to(dev)

    net = Net(len(CLASSES)).to(dev)
    opt = torch.optim.AdamW(net.parameters(), 1e-3, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, a.epochs)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    best = -1
    for ep in range(a.epochs):
        net.train()
        for x, y in tl:
            opt.zero_grad(); loss_fn(net(x.to(dev)), y.to(dev)).backward(); opt.step()
        sched.step()
        y, p = evaluate(net, vl, dev); f1 = macro_f1(y, p, len(CLASSES))
        print(f"Epoch {ep+1}/{a.epochs} | val accuracy {(y == p).mean():.3f} | macro F1 {f1:.3f}", flush=True)
        if f1 > best:
            best = f1; torch.save({"state": net.state_dict(), "classes": CLASSES}, a.out)
    print("best macroF1", best, "->", a.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prepare_data.py
import csv
import hashlib
import re
import shutil
import time
from collections import Counter
from itertools import combinations
from pathlib import Path, PurePosixPath
from urllib.request import Request, urlopen
from zipfile import BadZipFile, ZipFile, is_zipfile

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from tqdm.auto import tqdm

from common import CLASSES, POSE_WEIGHTS, T, action_window, frame_tracks, track_video

DATASET_URL = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "jonathannield/cctv-action-recognition-dataset?datasetVersionNumber=1"
)
FOLDER_LABEL = {
    "hit": "violence", "kick": "violence", "fall": "fall",
    "walk": "normal", "run": "normal", "sit": "normal", "stand": "normal",
}


def download_archive(path):
    if path.is_file() and is_zipfile(path):
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(".part")
    for attempt in range(3):
        try:
            with urlopen(Request(DATASET_URL), timeout=90) as response:
                size = int(response.headers.get("Content-Length", 0))
                with partial.open("wb") as out, tqdm(
                    total=size or None, unit="B", unit_scale=True, desc="Kaggle download"
                ) as progress:
                    while chunk := response.read(1024 * 1024):
                        out.write(chunk)
                        progress.update(len(chunk))
            if not is_zipfile(partial):
                raise BadZipFile("Kaggle did not return a complete ZIP archive.")
            partial.replace(path)
            return
        except (OSError, BadZipFile) as error:
            partial.unlink(missing_ok=True)
            if attempt == 2:
                raise RuntimeError(
                    "The Kaggle download failed. Rerun this cell to retry, or download "
                    f"version 1 manually and place the ZIP at {path}."
                ) from error
            time.sleep(2 ** attempt)


def select_clips(members):
    rows = []
    for name in sorted(members):
        path = PurePosixPath(name)
        folder = path.parent.name
        if path.suffix.lower() != ".mp4" or folder not in FOLDER_LABEL:
            continue
        match = re.fullmatch(rf"(.+)_{re.escape(folder)}_\d+", path.stem)
        if match is None:
            raise ValueError(f"Cannot determine the original recording for {name}.")
        rows.append({
            "member": name,
            "group": match[1],
            "label": FOLDER_LABEL[folder],
            "key": hashlib.sha256(name.encode()).hexdigest()[:20],
        })
    if set(row["label"] for row in rows) != set(CLASSES):
        raise ValueError("The archive does not contain all three expected action labels.")
    return rows


def assign_splits(rows):
    labels = [row["label"] for row in rows]
    groups = [row["group"] for row in rows]
    folds = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    holdouts = [valid.tolist() for _, valid in folds.split(labels, labels, groups)]
    totals = Counter(labels)

    def count(indices):
        return Counter(labels[i] for i in indices)

    def distance(counts):
        return sum(abs(counts[label] / totals[label] - 0.1) for label in CLASSES)

    candidates = []
    for val, test in combinations(holdouts, 2):
        val_counts, test_counts = count(val), count(test)
        train_counts = totals - val_counts - test_counts
        if all(counts[label] > 0 for counts in (train_counts, val_counts, test_counts)
               for label in CLASSES):
            candidates.append((distance(val_counts) + distance(test_counts), val, test))
    if not candidates:
        raise ValueError("Could not put every class in all three recording-level splits.")
    _, val, test = min(candidates, key=lambda candidate: candidate[0])
    val, test = set(val), set(test)
    for i, row in enumerate(rows):
        row["split"] = "val" if i in val else "test" if i in test else "train"


def prepare(workdir):
    from ultralytics import YOLO

    archive_path = workdir / "nield-v1.zip"
    download_archive(archive_path)
    model = YOLO(POSE_WEIGHTS)
    rows_used = []
    with ZipFile(archive_path) as archive:
        rows = select_clips(archive.namelist())
        assign_splits(rows)
        for row in tqdm(rows, desc="Extracting poses", unit="clip"):
            video = workdir / "videos" / (row["key"] + ".mp4")
            sample = workdir / "poses" / row["split"] / (row["key"] + ".npz")
            sample.parent.mkdir(parents=True, exist_ok=True)
            skipped = sample.with_suffix(".skip")
            if skipped.exists() and not sample.exists():
                continue
            if not video.exists():
                video.parent.mkdir(parents=True, exist_ok=True)
                temporary = video.with_suffix(".part")
                with archive.open(row["member"]) as source, temporary.open("wb") as target:
                    shutil.copyfileobj(source, target)
                temporary.replace(video)
            if not sample.exists():
                frames = [frame_tracks(result) for result in track_video(model, video, {})]
                start = max(0, (len(frames) - T) // 2)
                window = action_window(frames[start:start + T])
                if window is None:
                    skipped.touch()
                    continue
                temporary = sample.with_suffix(".part")
                with temporary.open("wb") as handle:
                    np.savez_compressed(handle, x=window, y=CLASSES.index(row["label"]))
                temporary.replace(sample)
            rows_used.append({**row, "video": str(video)})

    counts = Counter((row["split"], row["label"]) for row in rows_used)
    print(f"Usable clips: {len(rows_used)}/{len(rows)}")
    for split in ("train", "val", "test"):
        print(split, {label: counts[split, label] for label in CLASSES})
    if any(counts[split, label] == 0 for split in ("train", "val", "test") for label in CLASSES):
        raise RuntimeError("Pose extraction left at least one split without usable clips for a class.")
    with (workdir / "clips.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows_used[0]))
        writer.writeheader()
        writer.writerows(rows_used)

## 3. Download data and extract poses

The first run downloads the dataset and prepares one centered motion window per usable clip. All clips from the same original recording stay in the same train, validation, or test split. This keeps adjacent clips from leaking across splits.

In [ ]:
from pathlib import Path
from prepare_data import prepare

WORKDIR = Path("/content/zenith_cctv_v1")
prepare(WORKDIR)

## 4. Train and evaluate

In [ ]:
import subprocess
import sys

WEIGHTS = WORKDIR / "zenith_cctv_action.pt"
subprocess.run(
    [sys.executable, "-u", "train.py", "--data", str(WORKDIR / "poses"),
     "--epochs", str(EPOCHS), "--out", str(WEIGHTS)],
    check=True,
)

In [ ]:
import pandas as pd
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

from model import Net
from train import DS, evaluate

checkpoint = torch.load(WEIGHTS, map_location="cpu", weights_only=True)
classes = checkpoint["classes"]
net = Net(len(classes)).to("cuda")
net.load_state_dict(checkpoint["state"])
test_data = DS(WORKDIR / "poses", "test", False)
truth, predictions = evaluate(net, DataLoader(test_data, batch_size=64), "cuda")
label_ids = list(range(len(classes)))
print(classification_report(
    truth, predictions, labels=label_ids, target_names=classes, zero_division=0,
))
print("Confusion matrix (rows = actual, columns = predicted)")
display(pd.DataFrame(
    confusion_matrix(truth, predictions, labels=label_ids), index=classes, columns=classes,
))

## 5. Run a demo and download the model

The demo runs automatically on a held-out fight clip. By default it shows the highest-scoring window and a frame from that window. It flags suspected fights and falls when the top score reaches 0.60. To try another clip, upload it through the Colab Files panel, enter its path in the settings cell, rerun that cell, then rerun the demo cell below.

In [ ]:
# @title Run video demo
import csv
from collections import deque

from IPython.display import Image, display
from ultralytics import YOLO

from common import POSE_WEIGHTS, T, action_window, frame_tracks, track_video

if DEMO_VIDEO:
    demo_path = Path(DEMO_VIDEO)
    if not demo_path.is_file():
        raise FileNotFoundError(f"Upload your video in the Colab Files panel: {demo_path}")
else:
    with (WORKDIR / "clips.csv").open() as handle:
        example = next(row for row in csv.DictReader(handle)
                       if row["split"] == "test" and row["label"] == "violence")
    demo_path = Path(example["video"])
    print(f"Held-out example: {example['member']}")

def demo_windows(path):
    info = {}
    pose = YOLO(POSE_WEIGHTS)
    frames = deque(maxlen=T)
    count = 0
    for count, result in enumerate(track_video(pose, path, info), start=1):
        frames.append(frame_tracks(result))
        if count >= T and (count - T) % 16 == 0:
            yield list(frames), result.orig_img, count / info["sample_fps"]
    if count and (count < T or (count - T) % 16):
        yield list(frames), result.orig_img, count / info["sample_fps"]


best_score, best_event = 0.0, None
alert_count = 0
net.eval()
with torch.inference_mode():
    for frames, image, seconds in demo_windows(demo_path):
        window = action_window(frames)
        if window is None:
            continue
        scores = net(torch.from_numpy(window)[None].to("cuda")).softmax(-1)[0]
        label_id = int(scores.argmax())
        label, score = classes[label_id], float(scores[label_id])
        if label != "normal" and score >= 0.60:
            alert_count += 1
            print(f"{seconds:5.1f}s | Suspected {label} | score {score:.2f}")
        if score > best_score:
            best_score = score
            best_event = (label, image.copy(), seconds)

if best_event is None:
    print("The demo did not find enough tracked body keypoints to classify this clip.")
else:
    label, image, seconds = best_event
    print(f"Top window: {label} at {seconds:.1f}s | score {best_score:.2f}")
    print(f"Windows above the alert threshold: {alert_count}")
    import cv2
    ok, encoded = cv2.imencode(".jpg", image)
    if ok:
        display(Image(data=encoded.tobytes(), width=600))

In [ ]:
# @title Download the trained model
if DOWNLOAD_CHECKPOINT:
    from google.colab import files
    files.download(str(WEIGHTS))
else:
    print(f"Model is available in the Files panel at {WEIGHTS}.")